# Aerodynamic Optimization of NACA Airfoils with Surrogate Modeling (SMT)

This notebook shows how **surrogate modeling** plus **EGO (Efficient Global
Optimization)** finds an optimal airfoil while running CFD only a handful of
times. We use the [SMT](https://smt.readthedocs.io/) toolbox from
ISAE-SUPAERO and XFOIL as the aerodynamic evaluator.

## Problem

Find the 4-digit NACA parameters that **minimize Cd** for **Cl approx 0.5**
at **Re = 1e6**.

## Why a surrogate?

A CFD call (here XFOIL) is expensive. Instead of sweeping the design space
with thousands of evaluations, we fit a **Kriging** model that approximates
the aerodynamic response *and* reports its own uncertainty. **EGO** exploits
that: each iteration it proposes the point with the highest *Expected
Improvement* (a balance between exploiting good regions and exploring
uncertain ones) and only then spends one real CFD evaluation. The optimum is
reached in **tens** of evaluations instead of **thousands**.

**Design space**

| Variable | Meaning | Range |
|----------|---------|-------|
| `m` | max camber (1st NACA digit / 100) | [0.01, 0.09] |
| `p` | camber position (2nd digit / 10) | [0.10, 0.60] |
| `t` | max thickness (last 2 digits / 100) | [0.08, 0.20] |
| `alpha` | angle of attack [deg] | [0.0, 8.0] |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from smt.sampling_methods import LHS
from smt.surrogate_models import KRG
from smt.applications import EGO
from smt.design_space import DesignSpace

from airfoil_utils import (
    XLIMITS, RANDOM_STATE, REYNOLDS, naca4, eval_xfoil,
)

# Reproducibility: fixed seed everywhere.
np.random.seed(RANDOM_STATE)

CL_TARGET, CL_TOL, PENALTY, CD_FAIL = 0.5, 0.05, 1000.0, 9999.0
N_SAMPLES, N_INFILL = 80, 30
print('Setup ready. Design bounds:')
print(XLIMITS)

## 1. Initial dataset (DOE) with LHS + XFOIL

Latin Hypercube Sampling spreads 80 points evenly across the 4D space. Each
is turned into a NACA geometry and evaluated with XFOIL. Non-converged cases
are dropped (and counted).

In [ ]:
sampling = LHS(xlimits=XLIMITS, seed=RANDOM_STATE)
x_doe = sampling(N_SAMPLES)

records, failures = [], 0
for m, p, t, alpha in x_doe:
    res = eval_xfoil(m, p, t, alpha, reynolds=REYNOLDS)
    if res is None:
        failures += 1
        continue
    cl, cd, cm = res
    records.append([m, p, t, alpha, cl, cd, cm])

df = pd.DataFrame(records, columns=['m','p','t','alpha','Cl','Cd','Cm'])
df.to_csv('data/airfoil_dataset.csv', index=False)
print(f'Converged: {len(df)} / {N_SAMPLES}   (dropped: {failures})')
df.head()

## 2. Surrogate training (Kriging) + Leave-One-Out validation

Two independent Kriging models with a Matern 5/2 kernel: one for `Cl`, one
for `Cd`. We measure generalization with Leave-One-Out CV (RMSE and R2).

In [ ]:
def build_krg():
    return KRG(corr='matern52', poly='constant', theta0=[1e-2],
               print_global=False, seed=RANDOM_STATE)

def train_model(x, y):
    sm = build_krg(); sm.set_training_values(x, y); sm.train(); return sm

def leave_one_out(x, y):
    n = x.shape[0]; pred = np.zeros(n)
    for i in range(n):
        mask = np.ones(n, bool); mask[i] = False
        sm = train_model(x[mask], y[mask])
        pred[i] = sm.predict_values(x[i:i+1])[0, 0]
    return pred

def metrics(yt, yp):
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    r2 = float(1 - np.sum((yt-yp)**2) / np.sum((yt-yt.mean())**2))
    return rmse, r2

X = df[['m','p','t','alpha']].values
y_cl, y_cd = df['Cl'].values, df['Cd'].values

sm_cl = train_model(X, y_cl.reshape(-1,1))
sm_cd = train_model(X, y_cd.reshape(-1,1))

cl_loo = leave_one_out(X, y_cl.reshape(-1,1))
cd_loo = leave_one_out(X, y_cd.reshape(-1,1))
rmse_cl, r2_cl = metrics(y_cl, cl_loo)
rmse_cd, r2_cd = metrics(y_cd, cd_loo)
print(f'Cl : RMSE={rmse_cl:.5f}  R2={r2_cl:.4f}')
print(f'Cd : RMSE={rmse_cd:.6f}  R2={r2_cd:.4f}')

### Surrogate validation plot (predicted vs real, Leave-One-Out)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yt, yp, lab, r2, rmse in [
    (axes[0], y_cl, cl_loo, 'Cl', r2_cl, rmse_cl),
    (axes[1], y_cd, cd_loo, 'Cd', r2_cd, rmse_cd)]:
    ax.scatter(yt, yp, s=22, edgecolors='k', linewidths=0.3, alpha=0.8)
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'r--', label='Ideal (y=x)')
    ax.set_xlabel(f'Real {lab}'); ax.set_ylabel(f'Predicted {lab}')
    ax.set_title(f'{lab}: R2={r2:.4f}  RMSE={rmse:.5f}'); ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Optimization with EGO

Objective: minimize `Cd`, with the `Cl approx 0.5` band enforced as a soft
penalty `f = Cd + 1000 * max(0, |Cl - 0.5| - 0.05)`. EGO reuses the DOE as
its starting set and spends 30 extra XFOIL evaluations (infill points). If
XFOIL fails on a point we assign `Cl=0, Cd=9999` and continue.

In [ ]:
_log = {'cl': [], 'cd': [], 'f': []}

def objective(x):
    n = x.shape[0]; y = np.zeros((n, 1))
    for i in range(n):
        m, p, t, alpha = x[i]
        res = eval_xfoil(m, p, t, alpha)
        cl, cd = (0.0, CD_FAIL) if res is None else (res[0], res[1])
        f = cd + PENALTY * max(0.0, abs(cl - CL_TARGET) - CL_TOL)
        y[i, 0] = f
        _log['cl'].append(cl); _log['cd'].append(cd); _log['f'].append(f)
    return y

f_doe = y_cd + PENALTY * np.maximum(0.0, np.abs(y_cl - CL_TARGET) - CL_TOL)

ego = EGO(n_iter=N_INFILL, criterion='EI', xdoe=X, ydoe=f_doe.reshape(-1,1),
          surrogate=KRG(corr='matern52', print_global=False,
                        design_space=DesignSpace(XLIMITS),
                        seed=RANDOM_STATE),
          seed=RANDOM_STATE)

x_opt, y_opt, _, x_data, y_data = ego.optimize(fun=objective)
m_o, p_o, t_o, a_o = x_opt
cl_o, cd_o, _ = eval_xfoil(m_o, p_o, t_o, a_o)
desig = f'NACA {int(round(m_o*100))}{int(round(p_o*10))}{int(round(t_o*100)):02d}'
print(f'Optimal: {desig}')
print(f'  m={m_o:.4f} p={p_o:.4f} t={t_o:.4f} alpha={a_o:.3f} deg')
print(f'  Cl={cl_o:+.4f}  Cd={cd_o:.6f}')

### Convergence: EGO vs random search

Same infill points, but the random baseline consumes them in shuffled order.
EGO front-loads the best designs because it chooses *where* to look.

In [ ]:
infill_f = np.array(_log['f'][:N_INFILL])
infill_cd = np.array(_log['cd'][:N_INFILL])

def running_best(order):
    bf, bcd = f_doe.min(), y_cd[np.argmin(f_doe)]
    out = []
    for k in order:
        if infill_f[k] < bf:
            bf, bcd = infill_f[k], infill_cd[k]
        out.append(bcd)
    return np.array(out)

conv = running_best(range(N_INFILL))
perm = np.random.default_rng(RANDOM_STATE).permutation(N_INFILL)
conv_rand = running_best(perm)

ev = np.arange(1, N_INFILL + 1)
plt.figure(figsize=(8, 5))
plt.plot(ev, conv, '-o', label='EGO (Expected Improvement)')
plt.plot(ev, conv_rand, '--s', ms=4, label='Random search (same points)')
plt.xlabel('CFD evaluations (infill)'); plt.ylabel('Best feasible Cd')
plt.title('Convergence: EGO vs random search')
plt.grid(True, alpha=0.3); plt.legend(); plt.show()

## 4. Response surface and optimal geometry

In [ ]:
t_grid = np.linspace(0.08, 0.20, 60); a_grid = np.linspace(0.0, 8.0, 60)
T, A = np.meshgrid(t_grid, a_grid)
pts = np.column_stack([np.full(T.size, m_o), np.full(T.size, p_o),
                       T.ravel(), A.ravel()])
cd_pred = sm_cd.predict_values(pts).reshape(T.shape)
cd_std = np.sqrt(np.abs(sm_cd.predict_variances(pts))).reshape(T.shape)

plt.figure(figsize=(8, 6))
cf = plt.contourf(T, A, cd_pred, levels=30, cmap='viridis')
plt.colorbar(cf, label='Predicted Cd (Kriging)')
cu = plt.contour(T, A, cd_std, levels=6, colors='white', alpha=0.5)
plt.clabel(cu, inline=True, fontsize=7, fmt='std=%.4f')
plt.scatter(df['t'], df['alpha'], c='red', s=18, edgecolors='k',
            linewidths=0.4, label='Training points')
plt.scatter([t_o], [a_o], marker='*', s=320, c='gold', edgecolors='k',
            label='EGO optimum')
plt.xlabel('Thickness t'); plt.ylabel('alpha [deg]')
plt.title(f'Cd response surface (m={m_o:.3f}, p={p_o:.3f} fixed)')
plt.legend(loc='upper left'); plt.show()

In [ ]:
xg, yg = naca4(m_o, p_o, t_o)
plt.figure(figsize=(9, 3.2))
plt.plot(xg, yg, '-', lw=1.8); plt.fill(xg, yg, alpha=0.10)
plt.axhline(0, color='gray', lw=0.5, ls=':')
plt.gca().set_aspect('equal')
plt.title(f'Optimal airfoil: {desig}')
plt.xlabel('x/c'); plt.ylabel('y/c'); plt.show()

## Conclusions

EGO located the optimum using only the real CFD evaluations below. Compare
that with an equivalent brute-force grid search over the same 4 variables.

In [ ]:
used = len(df) + N_INFILL
grid_per_dim = 10
grid_total = grid_per_dim ** 4
print(f'Real CFD evaluations used (DOE + EGO): {used}')
print(f'Equivalent grid search ({grid_per_dim} pts/dim, 4 dims): {grid_total}')
print(f'Speed-up factor: ~{grid_total / used:.0f}x fewer evaluations')
print(f'\nBest airfoil: {desig} | Cl={cl_o:+.3f} Cd={cd_o:.5f}')